# Notebook 32 — Final IMERG precipitation-convergence analysis

This is Phase 3. It makes no ERA5 or NASA network requests. It reads the compact event data backed up by Notebook 31, verifies that every event available in IMERG Final V07 is complete, then creates the four scatterplots and the correlation/regression table. Events outside the current Final-V07 archive are retained in the inventory as documented exclusions and are not mixed with a different IMERG product.

The predictor is the saved convergence for the digitized Shinoda JPCZ polygon. The two precipitation regions are the same Shinoda polygon and the additional coastal wedge.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = 'codex/notebook16-pcolormesh'
REPO_DIR = '/content/JPCZcatalog'
FORCE_REFRESH_REPO = True
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
# Accept a raw Git URL even if it was accidentally pasted as a Markdown link.
if REPO_URL.startswith('[') and '](' in REPO_URL and REPO_URL.endswith(')'):
    REPO_URL = REPO_URL.rsplit('](', 1)[1][:-1]
if not REPO_URL.startswith('https://'):
    raise ValueError(f'REPO_URL must be a raw https Git URL, not {REPO_URL!r}')

from google.colab import drive
drive.mount('/content/drive')
if FORCE_REFRESH_REPO and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
if not Path(REPO_DIR).exists():
    clone = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], text=True, capture_output=True)
    if clone.returncode:
        raise RuntimeError(f'Git clone failed for {REPO_URL}:\n{clone.stderr}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from jpcz_catalog.imerg_workflow import association_statistics, atomic_csv, read_checkpoint, write_event_plan

ALLOW_PARTIAL_ANALYSIS = False
DRIVE_ANALYSIS_DIR = DRIVE_ROOT / 'imerg_precipitation_convergence'
PLAN_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_collection_plan.csv'
EVENT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_regional_precipitation.csv'
ANALYSIS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_precipitation_convergence_metrics.csv'
STATS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_statistics.csv'
FIELD_GUIDE_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_statistics_field_guide.csv'
PLOT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_scatter.png'

if not PLAN_PATH.exists():
    raise FileNotFoundError('The collection plan is missing. Run Notebook 30, then collect data with Notebook 31.')
plan = pd.read_csv(PLAN_PATH, parse_dates=['event_start', 'event_end', 'event_peak', 'precip_window_start', 'precip_window_end_exclusive'])
event_metrics = read_checkpoint(EVENT_PATH, parse_dates=('event_peak',))
if not {'event_id', 'event_peak'}.issubset(event_metrics.columns):
    event_metrics = pd.DataFrame(columns=['event_id', 'event_peak'])
# Apply the current Final-V07 availability policy even if this Drive plan was created by an older notebook version.
plan = write_event_plan(plan, event_metrics, path=PLAN_PATH)
inventory = plan.merge(event_metrics, on=['event_id', 'event_peak'], how='left')
atomic_csv(inventory, ANALYSIS_PATH)
available_plan = plan.loc[plan['analysis_inclusion'].eq('include')].copy()
analysis = inventory.loc[inventory['analysis_inclusion'].eq('include')].copy()
complete_events = int((available_plan['collection_status'] == 'complete').sum())
unavailable_events = int((plan['analysis_inclusion'] == 'exclude').sum())
all_data_ready = complete_events == len(available_plan) and len(available_plan) > 0
print(f'Final V07 analysis inventory: {complete_events}/{len(available_plan)} available events complete; {unavailable_events} catalog events excluded because Final V07 is unavailable for their windows.')
print('Final-analysis readiness:', 'READY' if all_data_ready else 'NOT READY — resume Notebook 31.')
display(plan['collection_status'].value_counts().rename_axis('status').reset_index(name='event_count'))
display(analysis.head())

In [ ]:
specifications = []
for region, label in [('jpcz_polygon', 'Shinoda JPCZ polygon'), ('coastal_wedge', 'Coastal wedge')]:
    specifications.extend([
        (label, 'IMERG accumulation (mm)', 'jpcz_polygon_convergence_1e5_s-1', f'{region}_imerg_accumulation_mm'),
        (label, 'IMERG mean rate (mm h-1)', 'jpcz_polygon_convergence_1e5_s-1', f'{region}_imerg_mean_rate_mm_hr'),
    ])

if not all_data_ready and not ALLOW_PARTIAL_ANALYSIS:
    statistics_table = pd.DataFrame([
        {'region': region, 'precipitation_measure': measure, 'n': 0, 'status': 'waiting for complete Drive inventory'}
        for region, measure, _, _ in specifications
    ])
    print('Final statistics remain locked until all Final-V07-available events are saved.')
else:
    statistics_table = pd.DataFrame([
        association_statistics(analysis, x_column=x, y_column=y, region=region, measure=measure)
        for region, measure, x, y in specifications
    ])
atomic_csv(statistics_table, STATS_PATH)
presentation_columns = [
    'region', 'precipitation_measure', 'n',
    'x_mean', 'x_sample_sd', 'y_mean', 'y_sample_sd',
    'pearson_r', 'r_95ci_low', 'r_95ci_high', 'r_two_sided_p',
    'slope', 'slope_standard_error', 'slope_95ci_low', 'slope_95ci_high',
    'intercept', 'intercept_standard_error', 'r_squared',
    'rmse', 'residual_standard_error', 'regression_df',
    'evidence_for_nonzero_association_alpha_0.05', 'status',
]
field_guide = pd.DataFrame([
    ('x_mean / x_sample_sd', 'Mean and n−1 sample SD of convergence strength C = −D12 (10^-5 s^-1).'),
    ('y_mean / y_sample_sd', 'Mean and n−1 sample SD of the stated precipitation response; its units follow the response label.'),
    ('r_95ci_low / high', '95% Pearson-r confidence interval from Fisher z; SE_z = 1/sqrt(n−3).'),
    ('slope_95ci_low / high', '95% OLS slope interval: slope ± t_(0.975, n−2) × slope standard error.'),
    ('rmse', 'Root mean squared vertical residual; in the same units as the precipitation response.'),
    ('residual_standard_error', 'Residual SD estimated with n−2 regression degrees of freedom.'),
], columns=['field', 'definition'])
atomic_csv(field_guide, FIELD_GUIDE_PATH)
print('Detailed statistics saved:', STATS_PATH)
print('Field definitions saved:', FIELD_GUIDE_PATH)
display(statistics_table.reindex(columns=presentation_columns).round(4))
display(field_guide)

In [ ]:
def plot_association(ax, x_column, y_column, title, ylabel, summary):
    if summary.get('status') != 'ok':
        ax.text(0.5, 0.5, 'Waiting for complete IMERG event inventory', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return None
    sample = analysis[[x_column, y_column, 'duration_hours']].dropna()
    points = ax.scatter(sample[x_column], sample[y_column], c=sample['duration_hours'], cmap='viridis', s=40, alpha=0.85, edgecolor='white', linewidth=0.35)
    fit = stats.linregress(sample[x_column], sample[y_column])
    xline = np.linspace(sample[x_column].min(), sample[x_column].max(), 100)
    ax.plot(xline, fit.intercept + fit.slope * xline, color='#c0392b', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('925-hPa convergence strength, C = −D12 (10^-5 s^-1)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    response_units = 'mm' if 'accumulation' in ylabel.lower() else 'mm h^-1'
    statistics_text = (
        f"Events: n = {int(summary['n'])}\n"
        f"Pearson r (linear association): {summary['pearson_r']:+.2f}\n"
        f"95% CI for r: {summary['r_95ci_low']:+.2f} to {summary['r_95ci_high']:+.2f}\n"
        f"OLS slope: {summary['slope']:+.2f} {response_units} per 10^-5 s^-1\n"
        f"95% CI for slope: {summary['slope_95ci_low']:+.2f} to {summary['slope_95ci_high']:+.2f}\n"
        f"Typical scatter about line (RMSE): {summary['rmse']:.2f} {response_units}\n"
        f"Two-sided p for no linear association: {summary['r_two_sided_p']:.3g}"
    )
    ax.text(0.03, 0.97, statistics_text, va='top', transform=ax.transAxes, fontsize=7.0, linespacing=1.25, bbox={'facecolor': 'white', 'alpha': 0.94, 'edgecolor': '#555555'})
    return points

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
last_points = None
for ax, (region, measure, x_column, y_column) in zip(axes.flat, specifications):
    summary = statistics_table.loc[(statistics_table['region'] == region) & (statistics_table['precipitation_measure'] == measure)].iloc[0].to_dict()
    result = plot_association(ax, x_column, y_column, f'{region}: {measure}', measure, summary)
    if result is not None:
        last_points = result
if last_points is not None:
    fig.colorbar(last_points, ax=axes, shrink=0.82, pad=0.02, label='Merged-event duration (h)')
fig.suptitle('IMERG Final V07 precipitation versus JPCZ convergence strength (C = −D12)\nPositive r or slope means stronger convergence is associated with greater precipitation.', fontsize=14)
fig.text(0.5, 0.01, 'Key: r is the unitless Pearson linear correlation; its 95% CI is the uncertainty range for the population correlation. The slope gives the mean precipitation change for a 1 × 10^-5 s^-1 increase in convergence strength. RMSE is the typical vertical scatter of events around the red regression line.', ha='center', va='bottom', fontsize=7.7, wrap=True)
fig.savefig(PLOT_PATH, dpi=220, bbox_inches='tight')
plt.show()
print('Saved figure:', PLOT_PATH)

## Methods wording

For each merged JPCZ episode, the catalog retains the detector's saved 12-hour trailing, area-weighted 925-hPa divergence at the event peak, denoted D12. In the detector and catalog, D12 < 0 is convergence and D12 > 0 is divergence. For the association plots and regression only, we define convergence strength as C = −D12, so larger plotted values mean stronger convergence; the detector and its thresholds remain based on unmodified divergence. We obtained GPM IMERG Final V07 gauge-calibrated precipitation (`precipitation`; half-hourly 0.1 degree grid, with a legacy `precipitationCal` fallback only if present), calculated cosine-latitude-area-weighted precipitation rates over the Shinoda polygon and coastal wedge, calculated event accumulation by summing rate times 0.5 hour, and calculated event mean rate by dividing by event-window duration. For each association, we report the mean and n−1 sample SD of both variables, two-sided Pearson r, a 95% r interval based on the Fisher-z approximation (SE_z = 1/sqrt(n−3)), and ordinary least-squares slope, slope SE, slope 95% interval based on t_(0.975, n−2), R², residual standard error, RMSE, and two-sided p. The test evaluates whether the population linear association is distinguishable from zero at α = 0.05; it does not establish causation.